In [0]:
import os
import requests
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
from pyspark.sql.functions import current_timestamp, lit

# SETUP LOCAL RAW DIRECTORIES
today_str = datetime.now().strftime('%Y-%m-%d')
base_raw_path = f"/tmp/fhir_raw_xml/{today_str}"

resources = ["Patient", "Encounter", "Observation", "Condition"]

for resource in resources:
    dir_path = os.path.join(base_raw_path, resource)
    os.makedirs(dir_path, exist_ok=True)

print(f"Local XML raw directories created successfully at: {base_raw_path}/")

In [0]:

# XML INGESTION FUNCTION
def fetch_and_ingest_fhir_xml(resource_name, days_back=3):
    start_date = (datetime.now() - timedelta(days=days_back)).strftime('%Y-%m-%d')
    base_url = f"https://hapi.fhir.org/baseR4/{resource_name}"
    
    # Pass _format=xml parameter to HAPI FHIR API
    url = f"{base_url}?_format=xml&_lastUpdated=ge{start_date}&_count=100"
    
    all_xml_entries = []

    print(f"Processing Resource (XML): {resource_name} (Days Back: {days_back})")

    # Incremental API Fetch with XML Pagination
    while url:
        try:
            response = requests.get(url, timeout=30)
            if response.status_code != 200:
                print(f"Error fetching {url}: Status {response.status_code}")
                break

            xml_content = response.text
            
            # Parse XML response using ElementTree
            root = ET.fromstring(xml_content)
            
            # Extract individual resource XML strings from FHIR Bundle entries
            # FHIR XML namespaces: http://hl7.org/fhir
            ns = {'fhir': 'http://hl7.org/fhir'}
            
            for entry in root.findall('fhir:entry', ns):
                resource_elem = entry.find('fhir:resource', ns)
                if resource_elem is not None and len(resource_elem) > 0:
                    # Convert XML node back to raw string
                    inner_xml = ET.tostring(resource_elem[0], encoding='unicode')
                    all_xml_entries.append(inner_xml)

            # Find next page link in FHIR XML bundle
            next_link = None
            for link in root.findall('fhir:link', ns):
                relation = link.find('fhir:relation', ns)
                if relation is not None and relation.attrib.get('value') == 'next':
                    url_elem = link.find('fhir:url', ns)
                    if url_elem is not None:
                        next_link = url_elem.attrib.get('value')
                        break
            
            url = next_link

        except Exception as e:
            print(f"Error during XML pagination: {e}")
            break

    print(f"Retrieved {len(all_xml_entries)} XML records for {resource_name}.")
    if not all_xml_entries:
        return

    # STORE RAW XML LOCALLY (Metadata Archiving)
    local_dir_path = f"/tmp/fhir_raw_xml/{today_str}/{resource_name}"
    os.makedirs(local_dir_path, exist_ok=True)
    
    local_file_path = os.path.join(local_dir_path, f"{resource_name}_raw.xml")
    
    # Save combined XML payload
    with open(local_file_path, "w", encoding="utf-8") as f:
        combined_xml = "<Bundle>\n" + "\n".join(all_xml_entries) + "\n</Bundle>"
        f.write(combined_xml)

    # SPARK XML PARSING & DELTA WRITE
    # Convert XML strings into a PySpark DataFrame
    xml_tuples = [(x,) for x in all_xml_entries]
    df_raw_str = spark.createDataFrame(xml_tuples, ["raw_xml"])

    # Parse XML raw string into structured PySpark columns
    df_parsed = spark.read.format("xml") \
        .option("rowTag", resource_name) \
        .xml(df_raw_str.rdd.map(lambda r: r.raw_xml))

    # Save Raw Table
    raw_table_name = f"raw.{resource_name.lower()}_xml"
    df_parsed.write.format("delta").mode("overwrite").saveAsTable(raw_table_name)

    # Save Bronze Table with Ingestion Audit Metadata
    df_bronze = df_parsed.withColumn("extraction_timestamp", current_timestamp()) \
                         .withColumn("api_url_or_params", lit(f"{base_url}?_format=xml&_lastUpdated=ge{start_date}"))

    bronze_table_name = f"bronze.{resource_name.lower()}_xml"
    df_bronze.write.format("delta").mode("overwrite").saveAsTable(bronze_table_name)
    
    print(f"Successfully ingested & logged XML {resource_name} into {raw_table_name} & {bronze_table_name}")

# DYNAMIC METADATA-DRIVEN EXECUTION LOOP
metadata_df = spark.sql("""
    SELECT resource_name, days_back
    FROM raw.ingestion_metadata
    WHERE is_active = true
    ORDER BY execution_order ASC
""").collect()

for row in metadata_df:
    fetch_and_ingest_fhir_xml(resource_name=row["resource_name"], days_back=row["days_back"])